# M13 Lab — Agentic DS Workflows

**Datasets:** `housing_prices.csv`, `ecommerce_orders.csv` &nbsp;|&nbsp; **Focus:** ReAct loop, tool design, verification


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def load_csv(path: str) -> pd.DataFrame:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    return pd.read_csv(file_path)

def describe_dataframe(path: str) -> dict:
    df = load_csv(path)
    return {
        "shape": df.shape,
        "columns": list(df.columns),
        "missing": df.isna().sum().to_dict(),
        "numeric_summary": df.describe(include="number").to_dict(),
    }

def plot_histogram(path: str, column: str) -> dict:
    df = load_csv(path)
    if column not in df.columns:
        raise KeyError(f"Column not found: {column}")
    ax = df[column].dropna().plot(kind="hist", bins=20, title=f"Histogram of {column}")
    plt.show()
    return {
        "column": column,
        "mean": float(df[column].mean()),
        "median": float(df[column].median()),
        "non_null": int(df[column].notna().sum()),
    }


In [ ]:
import json
import re
import requests

def call_ollama(messages, model="gemma4n"):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

def parse_action(text: str):
    final_match = re.search(r"Final Answer:\s*(.*)", text, flags=re.S)
    if final_match:
        return {"type": "final", "content": final_match.group(1).strip()}
    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match = re.search(r"Action Input:\s*(\{.*\})", text, flags=re.S)
    if not action_match or not input_match:
        raise ValueError(f"Could not parse agent output: {text}")
    return {
        "type": "action",
        "action": action_match.group(1),
        "input": json.loads(input_match.group(1)),
    }

def run_agent(goal: str, tools: dict, max_iters: int = 4):
    tool_docs = "\n".join(f"- {name}: {func.__doc__ or 'No docstring provided.'}" for name, func in tools.items())
    messages = [{
        "role": "system",
        "content": (
            "You are a local ReAct DS agent. Use one tool at a time. "
            "Respond with either:\nAction: <tool_name>\nAction Input: <JSON>\n"
            "or Final Answer: <grounded summary>.\n"
            f"Available tools:\n{tool_docs}"
        ),
    }, {"role": "user", "content": goal}]
    trace = []
    for _ in range(max_iters):
        agent_text = call_ollama(messages)
        trace.append(agent_text)
        decision = parse_action(agent_text)
        if decision["type"] == "final":
            return {"final_answer": decision["content"], "trace": trace}
        tool_name = decision["action"]
        observation = tools[tool_name](**decision["input"])
        messages.append({"role": "assistant", "content": agent_text})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return {"final_answer": "Stopped: max iterations reached.", "trace": trace}


In [ ]:
        tools = {
            "describe_dataframe": describe_dataframe,
            "plot_histogram": plot_histogram,
        }

        result = run_agent(
            goal=(
                "Explore housing_prices.csv. First inspect the dataset structure. "
                "Then decide whether one histogram would help. "
                "End with Final Answer containing 2 grounded findings."
            ),
            tools=tools,
            max_iters=4,
        )

        print("FINAL ANSWER:
", result["final_answer"])
        print("
TRACE:")
        for item in result["trace"]:
            print("-" * 60)
            print(item)


## L13.5 [AI-OFF] — Verification exercise

Without using any AI help, inspect the agent trace from the previous cell.
In a fresh code or markdown cell, do all of the following:
1. identify one claim that needs manual recomputation,
2. recompute it directly from the dataset,
3. state whether the agent's wording was justified,
4. name one improvement you would make to the toolset or stopping rules.
